# global_config\n
Generated from 00_common/global_config.py for Databricks notebook import.\n

In [ ]:
# Global Configuration Loader for Databricks Ingestion Framework
"""
Loads global YAML configuration with environment variable substitution and validation.
"""

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Any
import yaml

_ENV_PATTERN = re.compile(r"\$\{([A-Z0-9_]+)(?::([^}]*))?\}")

def _substitute_env(value: str) -> str:
    def replacer(match: re.Match[str]) -> str:
        key = match.group(1)
        default_value = match.group(2) if match.group(2) is not None else ""
        return os.getenv(key, default_value)
    return _ENV_PATTERN.sub(replacer, value)

def _resolve_obj(obj: Any) -> Any:
    if isinstance(obj, str):
        return _substitute_env(obj)
    if isinstance(obj, dict):
        return {k: _resolve_obj(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_resolve_obj(v) for v in obj]
    return obj

def load_global_config(config_path: str) -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Global config not found: {path}. "
            "Set --global-config to the correct path or check your deployment."
        )
    try:
        with path.open("r", encoding="utf-8") as f:
            payload = yaml.safe_load(f) or {}
    except yaml.YAMLError as exc:
        raise ValueError(f"Invalid YAML in global config {path}: {exc}") from exc
    if "lifecycle_log_table" not in payload:
        payload["lifecycle_log_table"] = "audit.entity_lifecycle_log"
    return _resolve_obj(payload)

def require_config_keys(config: dict, required_dotted_paths: list[str]) -> None:
    """Raise ValueError listing every dotted key path that is missing or empty."""
    missing: list[str] = []
    for dotted in required_dotted_paths:
        parts = dotted.split(".")
        node: Any = config
        for part in parts:
            if not isinstance(node, dict) or part not in node:
                missing.append(dotted)
                break
            node = node[part]
        else:
            if not node:
                missing.append(dotted)
    if missing:
        raise ValueError(f"Missing required global config values: {missing}")